In [1]:
import scarf
scarf.configure_output(level="WARNING", progress=False)

dataset = scarf.cytebase.connect("scarf_docs").download_dataset(
    "tenx_5K_pbmc_rnaseq",
    destination="scarf_datasets",
    zarr=True,
)
ds = scarf.DataStore(f"{dataset}/data.zarr", nthreads=4)
baseline_run = ds.pipeline.open(label="docs_default")
cell_selection = baseline_run["analysis_cell_selection"]
hvg_ref = baseline_run["highly_variable_features"]

In [2]:
normalized = baseline_run["normalized"]
pca = baseline_run["pca"]
ann = baseline_run["ann_index"]
neighbors_k11 = baseline_run["neighbors"]
graph_k11 = baseline_run["connectivity_map"]

In [3]:
[reopened_graph] = ds.list_artifacts(
    from_assay="RNA",
    kind="connectivity_map",
    operation="build_connectivity_map",
    inputs={"neighbors": neighbors_k11},
    complete_only=True,
)
assert reopened_graph == graph_k11

status = ds.inspect_artifact(reopened_graph)
{
    "operation": status.operation,
    "parameters": status.parameters,
    "inputs": status.inputs,
    "complete": status.complete,
}

{'operation': 'build_connectivity_map',
 'parameters': {'local_connectivity': 1.0, 'bandwidth': 1.5},
 'inputs': {'neighbors': {'type': 'artifact',
   'scope': 'assay',
   'kind': 'neighbors',
   'artifact_id': 'ff42aae647b32ffc1d2e43d3824a8cfd1e429d752a9b2b7a58942fbd19118d3a',
   'assay': 'RNA'}},
 'complete': True}

In [4]:
neighbors_k15 = ds.query_neighbors(ann, k=15)
graph_k15 = ds.build_connectivity_map(neighbors_k15)

{
    "normalization reused": ds.run_normalization(cell_selection, hvg_ref) == normalized,
    "PCA reused": ds.run_pca(normalized, dims=15) == pca,
    "ANN index reused": ds.build_ann_index(pca) == ann,
    "neighbors recomputed": neighbors_k15 != neighbors_k11,
    "graph recomputed": graph_k15 != graph_k11,
}

{'normalization reused': True,
 'PCA reused': True,
 'ANN index reused': True,
 'neighbors recomputed': True,
 'graph recomputed': True}

In [5]:
pca_dims20 = ds.run_pca(normalized, dims=20)
ann_dims20 = ds.build_ann_index(pca_dims20)
neighbors_dims20 = ds.query_neighbors(ann_dims20, k=11)
graph_dims20 = ds.build_connectivity_map(neighbors_dims20)

{
    "PCA recomputed": pca_dims20 != pca,
    "ANN index recomputed": ann_dims20 != ann,
    "neighbors recomputed": neighbors_dims20 != neighbors_k11,
    "graph recomputed": graph_dims20 != graph_k11,
}

{'PCA recomputed': True,
 'ANN index recomputed': True,
 'neighbors recomputed': True,
 'graph recomputed': True}

In [6]:
lineage = ds.lineage(
    {
        "k11 graph": graph_k11,
        "k15 graph": graph_k15,
        "dims20 graph": graph_dims20,
    }
)
lineage

```mermaid
flowchart LR
    artifact0["RNA / metadata_snapshot | snapshot_run_metadata | 38963facb6bb"]
    artifact1["datastore / cell_selection | snapshot_pipeline_input_selection | 211d197b723a"]
    artifact2["datastore / metadata_snapshot | snapshot_run_metadata | 59af88cbf4b0"]
    artifact3["datastore / cell_selection | filter_pipeline_cells | a4137c5ada6d"]
    artifact4["RNA / feature_summary | summarize_rna_features | 135f8d226824"]
    artifact5["RNA / feature_selection | select_hvgs | adeede36cdca"]
    artifact6["RNA / normalized | run_normalization | 24cfc31e5c04"]
    artifact7["RNA / feature_scaling | calculate_feature_scaling | 311f73798775"]
    artifact8["RNA / reduction | run_pca | 9f870f98ef66"]
    artifact9["RNA / ann_index | build_ann_index | 15c6e4c78333"]
    artifact10["RNA / neighbors | query_neighbors | 447a435bdda8"]
    artifact11["RNA / connectivity_map | build_connectivity_map | 88527b26accd | outputs: k15 graph"]
    artifact12["RNA / neighbors | query_neighbors | ff42aae647b3"]
    artifact13["RNA / connectivity_map | build_connectivity_map | d9a2230415db | outputs: k11 graph"]
    artifact14["RNA / reduction | run_pca | d3c0f8c2eef2"]
    artifact15["RNA / ann_index | build_ann_index | 68eba36a3eda"]
    artifact16["RNA / neighbors | query_neighbors | ff4d97a6e61f"]
    artifact17["RNA / connectivity_map | build_connectivity_map | d81f03684575 | outputs: dims20 graph"]
    artifact8 -->|"coordinates"| artifact10
    artifact9 -->|"ann_index"| artifact10
    artifact10 -->|"neighbors"| artifact11
    artifact8 -->|"coordinates"| artifact12
    artifact9 -->|"ann_index"| artifact12
    artifact12 -->|"neighbors"| artifact13
    artifact3 -->|"pca_cell_selection"| artifact14
    artifact6 -->|"normalized"| artifact14
    artifact7 -->|"feature_scaling"| artifact14
    artifact14 -->|"coordinates"| artifact15
    artifact14 -->|"coordinates"| artifact16
    artifact15 -->|"ann_index"| artifact16
    artifact16 -->|"neighbors"| artifact17
    artifact1 -->|"input_cell_selection"| artifact3
    artifact2 -->|"cell_snapshot"| artifact3
    artifact3 -->|"cell_selection"| artifact4
    artifact0 -->|"feature_snapshot"| artifact5
    artifact4 -->|"feature_summary"| artifact5
    artifact3 -->|"cell_selection"| artifact6
    artifact5 -->|"feature_selection"| artifact6
    artifact6 -->|"normalized"| artifact7
    artifact3 -->|"pca_cell_selection"| artifact8
    artifact6 -->|"normalized"| artifact8
    artifact7 -->|"feature_scaling"| artifact8
    artifact8 -->|"coordinates"| artifact9
```

### Artifact details

#### RNA / metadata_snapshot / 38963facb6bb
- Status: `complete`
- Path: `RNA/artifacts/metadata_snapshot/38963facb6bb61694cd6fe39abe760c77d036fccc519f0297801b7fcd8abd613`
- Operation: `snapshot_run_metadata`
- Parameters: `assay="RNA"; axis="feature"; ordered_columns=["names"]`
- Other inputs: `column_fingerprints={"names":"25b8d03a2f9b2b01183cc9742301f3cf9cd4762c4fe962fe3ad5189013d8c857"}; ordered_row_ids_fingerprint="4548c9820ab9ed0afbc48cbb6503bdcdc725b57c65dbfe3c4da8bb480b7fc4b1"`

#### datastore / cell_selection / 211d197b723a
- Status: `complete`
- Path: `artifacts/cell_selection/211d197b723a6ba786d94805e0a6f3f05741d2eaee5a072452f01cb0cd1eea5d`
- Operation: `snapshot_pipeline_input_selection`
- Parameters: `assay="RNA"`
- Execution options: `source_column="I"`
- Other inputs: `ordered_row_ids_fingerprint="f5f05615ffc8833f75752b75152ccd728702f42950f81e9c28402880a4dd2ae5"; values_fingerprint="94f668fa448559768ea628c836d7ed1b683636c280cd42e9a6c9b73ba0179a38"`

#### datastore / metadata_snapshot / 59af88cbf4b0
- Status: `complete`
- Path: `artifacts/metadata_snapshot/59af88cbf4b079de50ab013f04f7b0b862951cb7e4d3efaf47dea241171c9059`
- Operation: `snapshot_run_metadata`
- Parameters: `assay=null; axis="cell"; ordered_columns=["names","RNA_nCounts","RNA_nFeatures","RNA_percentMito"]`
- Other inputs: `column_fingerprints={"RNA_nCounts":"a8a7520000e32d28bcf97a8977290bcc7185570098e1fe95739c74687b435843","RNA_nFeatures":"a3df08addee7271194b0ebdfac85ad92ec93f3801031e65316776453eb...; ordered_row_ids_fingerprint="f5f05615ffc8833f75752b75152ccd728702f42950f81e9c28402880a4dd2ae5"`

#### datastore / cell_selection / a4137c5ada6d
- Status: `complete`
- Path: `artifacts/cell_selection/a4137c5ada6d3933e211d551dd636a097d6f0ed8103afde75fb3b2b05d67c16b`
- Operation: `filter_pipeline_cells`
- Parameters: `attrs=["RNA_nCounts","RNA_nFeatures","RNA_percentMito"]; enabled=true; highs=[15000,4000,15]; keepBounds=false; lows=[1000,500,0]; method="manual"`
- Execution options: `source_column="I"`
- Other inputs: `ordered_row_ids_fingerprint="f5f05615ffc8833f75752b75152ccd728702f42950f81e9c28402880a4dd2ae5"; values_fingerprint="6b282098296bdccef544dd1ce41e2f2987a3c75c74240296a5bbbd26e15eb00d"`

#### RNA / feature_summary / 135f8d226824
- Status: `complete`
- Path: `RNA/artifacts/feature_summary/135f8d226824425a58515e3da530d1f5c115e50031367124ea7ffd15a321b48d`
- Operation: `summarize_rna_features`
- Parameters: `normalization_method={"module":"scarf.assay","qualname":"norm_lib_size"}; size_factor=1000`
- Execution options: `nthreads=2`

#### RNA / feature_selection / adeede36cdca
- Status: `complete`
- Path: `RNA/artifacts/feature_selection/adeede36cdca822fde8cf2e62bb430843f28056a4e34414f753607c97a69240d`
- Operation: `select_hvgs`
- Parameters: `bin_strategy="adaptive"; blacklist="^MT-|^RPS|^RPL|^MRPS|^MRPL|^CCN|^HLA-|^H2-|^HIST|^XIST$|^DDX3Y$|^USP9Y$|^EIF1AY$|^KDM5D$|^SRY$|^ZFY$|^UTY$|^TMSB4Y$|^NLGN4Y$"; keep_bounds=false; lowess_frac=0.1; max_cells=3920; max_mean={"special_float":"inf"}; max_var={"special_float":"inf"}; min_cells=20; min_mean={"special_float":"-inf"}; min_var={"special_float":"-inf"}; n_bins=200; top_n=500`
- Execution options: `invalidate_cache=false; nthreads=2; plot_kwargs={}; show_plot=false`

#### RNA / normalized / 24cfc31e5c04
- Status: `complete`
- Path: `RNA/artifacts/normalized/24cfc31e5c045f8066a38de7ce90d4ca1c6fe7d9c140e6b6686c77e731fd75d0`
- Operation: `run_normalization`
- Parameters: `log_transform=true; normalization_method={"external_hook":true,"module":"scarf.assay","qualname":"norm_lib_size"}; renormalize_subset=true; size_factor=1000.0`
- Execution options: `invalidate_cache=false`
- Other inputs: `dataset_fingerprint="ac8346731fc57122b5a0d7b87e7639c5320157483505a3db4e30341b1f902729"`

#### RNA / feature_scaling / 311f73798775
- Status: `complete`
- Path: `RNA/artifacts/feature_scaling/311f737987759c3b4fe4450e62e813bf0ef9f30e8d735b804d1f779a89a62a13`
- Operation: `calculate_feature_scaling`
- Parameters: `enabled=true`
- Execution options: `batch_size=3940; invalidate_cache=false; local_cache="auto"`

#### RNA / reduction / 9f870f98ef66
- Status: `complete`
- Path: `RNA/artifacts/reduction/9f870f98ef66a62a250e123610d6433f786b242497a628fb59689c2322f5e1d8`
- Operation: `run_pca`
- Parameters: `dims=15; feat_scaling=true`
- Execution options: `batch_size=3940; invalidate_cache=false; local_cache="auto"; show_elbow_plot=false`

#### RNA / ann_index / 15c6e4c78333
- Status: `complete`
- Path: `RNA/artifacts/ann_index/15c6e4c783339fe01a68783538fc3926fb61199fa1ae2049fae9a9f595a27e3a`
- Operation: `build_ann_index`
- Parameters: `ann_ef=50; ann_efc=50; ann_m=48; ann_metric="l2"; ann_parallel=false; parallel_threads=null; rand_state=4466`
- Execution options: `batch_size=3940; invalidate_cache=false`

#### RNA / neighbors / 447a435bdda8
- Status: `complete`
- Path: `RNA/artifacts/neighbors/447a435bdda8e87ff74d5f909272fc61ffadecc8926f1cbe4be0adaa19a6dea2`
- Operation: `query_neighbors`
- Parameters: `distance_metric="l2"; k=15`
- Execution options: `batch_size=3940; invalidate_cache=false`

#### RNA / connectivity_map / 88527b26accd
- Status: `complete`
- Path: `RNA/artifacts/connectivity_map/88527b26accd72ac4f2963cc95bad6993fa0b8502fe263d18d1e2a5a46441bf9`
- Operation: `build_connectivity_map`
- Outputs: `k15 graph`
- Parameters: `bandwidth=1.5; local_connectivity=1.0`
- Execution options: `invalidate_cache=false`

#### RNA / neighbors / ff42aae647b3
- Status: `complete`
- Path: `RNA/artifacts/neighbors/ff42aae647b32ffc1d2e43d3824a8cfd1e429d752a9b2b7a58942fbd19118d3a`
- Operation: `query_neighbors`
- Parameters: `distance_metric="l2"; k=11`
- Execution options: `batch_size=3940; invalidate_cache=false`

#### RNA / connectivity_map / d9a2230415db
- Status: `complete`
- Path: `RNA/artifacts/connectivity_map/d9a2230415db3481cc9ecf80050c870d0ec2e2cef51fee26274da963d2c3fff1`
- Operation: `build_connectivity_map`
- Outputs: `k11 graph`
- Parameters: `bandwidth=1.5; local_connectivity=1.0`
- Execution options: `invalidate_cache=false`

#### RNA / reduction / d3c0f8c2eef2
- Status: `complete`
- Path: `RNA/artifacts/reduction/d3c0f8c2eef2e8de026d54a0181b6521a0f5df68a10cefce608fac3cc2a5c152`
- Operation: `run_pca`
- Parameters: `dims=20; feat_scaling=true`
- Execution options: `batch_size=3940; invalidate_cache=false; local_cache="auto"; show_elbow_plot=false`

#### RNA / ann_index / 68eba36a3eda
- Status: `complete`
- Path: `RNA/artifacts/ann_index/68eba36a3eda39dae6a8371f786a8083ca788205a16760948012a34e5bbaab59`
- Operation: `build_ann_index`
- Parameters: `ann_ef=50; ann_efc=50; ann_m=48; ann_metric="l2"; ann_parallel=false; parallel_threads=null; rand_state=4466`
- Execution options: `batch_size=3940; invalidate_cache=false`

#### RNA / neighbors / ff4d97a6e61f
- Status: `complete`
- Path: `RNA/artifacts/neighbors/ff4d97a6e61f4ff2a5bc32c94aa2e1baad048228e773fa8ed9d754ec213381f9`
- Operation: `query_neighbors`
- Parameters: `distance_metric="l2"; k=11`
- Execution options: `batch_size=3940; invalidate_cache=false`

#### RNA / connectivity_map / d81f03684575
- Status: `complete`
- Path: `RNA/artifacts/connectivity_map/d81f036845751fc7b2969b3c8ac6d2e763b8e7fb8a9ee50c54fb12091a02bef9`
- Operation: `build_connectivity_map`
- Outputs: `dims20 graph`
- Parameters: `bandwidth=1.5; local_connectivity=1.0`
- Execution options: `invalidate_cache=false`

In [7]:
lineage_markdown = lineage.to_markdown()
lineage_markdown.splitlines()[:12]

['```mermaid',
 'flowchart LR',
 '    artifact0["RNA / metadata_snapshot | snapshot_run_metadata | 38963facb6bb"]',
 '    artifact1["datastore / cell_selection | snapshot_pipeline_input_selection | 211d197b723a"]',
 '    artifact2["datastore / metadata_snapshot | snapshot_run_metadata | 59af88cbf4b0"]',
 '    artifact3["datastore / cell_selection | filter_pipeline_cells | a4137c5ada6d"]',
 '    artifact4["RNA / feature_summary | summarize_rna_features | 135f8d226824"]',
 '    artifact5["RNA / feature_selection | select_hvgs | adeede36cdca"]',
 '    artifact6["RNA / normalized | run_normalization | 24cfc31e5c04"]',
 '    artifact7["RNA / feature_scaling | calculate_feature_scaling | 311f73798775"]',
 '    artifact8["RNA / reduction | run_pca | 9f870f98ef66"]',
 '    artifact9["RNA / ann_index | build_ann_index | 15c6e4c78333"]']